In [24]:
import numpy as np
from QuditGates import QuditGates
from QuditCircuit import QuditCircuit
from helper_functions import *

## Initializations

In [25]:
dimension = 4
number_S_qudits = 2 # Number of S qudits in the circuit
number_qudits = 1 + 2*number_S_qudits # Total number of qudits in the circuit 

gates = QuditGates(dimension) # Initialize the QuditGtes class

We will initialize the qudit circuit as described in the paper.

In [26]:
qc = QuditCircuit(number_qudits,dimension)

print("Initial state of the circuit:")
qc.display_state()

Initial state of the circuit:
|00000> : (1+0j)


## Initialize the circuit

We follow the following structure of the qudits in the circuit

$
    q_0 \mapsto |\psi\rangle_A\\
    q_1 \mapsto |\psi\rangle_{S_1}\\
    q_2 \mapsto |\psi\rangle_{N_1}\\
    \dots\\
    q_{2n} \mapsto |\psi\rangle_{S_{n}}\\
    q_{2n+1} \mapsto |\psi\rangle_{N_{n}}\\
$


We initialize the data qudit $|\psi\rangle_{A}$ in the circuit.

In [27]:
qc.apply_gate(gates.get_I_gate(), [0]) # Apply identity gate to the first qudit

We initialize all the pairs of $(|\psi\rangle_{S_i},|\psi\rangle_{N_i})$ to the generalized Bell state $\frac{1}{\sqrt{d}}\sum_{p=0}^{d-1}|p\rangle|p\rangle$.

In [28]:
# Initialize the (S, N) pairs in the circuit
for i in range(number_S_qudits):
    qc.apply_gate(gates.get_H_gate(), [2*i+1])
    qc.apply_gate(gates.CSUM_gate(), [2*i+1, 2*i+2])

Display the quantum state

In [29]:
print("State of the operator circuit after initialization:")
qc.display_state_abs()

State of the operator circuit after initialization:
|00000> : 0.2500+0.0000j => |00000|=0.2500 => Prob=0.0625
|00011> : 0.2500+0.0000j => |00011|=0.2500 => Prob=0.0625
|00022> : 0.2500+0.0000j => |00022|=0.2500 => Prob=0.0625
|00033> : 0.2500+0.0000j => |00033|=0.2500 => Prob=0.0625
|01100> : 0.2500+0.0000j => |01100|=0.2500 => Prob=0.0625
|01111> : 0.2500+0.0000j => |01111|=0.2500 => Prob=0.0625
|01122> : 0.2500+0.0000j => |01122|=0.2500 => Prob=0.0625
|01133> : 0.2500+0.0000j => |01133|=0.2500 => Prob=0.0625
|02200> : 0.2500+0.0000j => |02200|=0.2500 => Prob=0.0625
|02211> : 0.2500+0.0000j => |02211|=0.2500 => Prob=0.0625
|02222> : 0.2500+0.0000j => |02222|=0.2500 => Prob=0.0625
|02233> : 0.2500+0.0000j => |02233|=0.2500 => Prob=0.0625
|03300> : 0.2500+0.0000j => |03300|=0.2500 => Prob=0.0625
|03311> : 0.2500+0.0000j => |03311|=0.2500 => Prob=0.0625
|03322> : 0.2500+0.0000j => |03322|=0.2500 => Prob=0.0625
|03333> : 0.2500+0.0000j => |03333|=0.2500 => Prob=0.0625


## Apply the encryption operator

In [30]:
targets = [0]+[2*i+1 for i in range(number_S_qudits)]
enc_matrix = gates.create_Uencryption(number_S_qudits)
qc.apply_gate(enc_matrix, targets=targets)

In [31]:
print("State of the operator circuit after encryption:")
qc.display_state_abs()

State of the operator circuit after encryption:
|00000> : 0.0884-0.0884j => |00000|=0.1250 => Prob=0.0156
|00011> : 0.1250+0.0000j => |00011|=0.1250 => Prob=0.0156
|00022> : -0.088+0.0884j => |00022|=0.1250 => Prob=0.0156
|00033> : 0.1250-0.0000j => |00033|=0.1250 => Prob=0.0156
|01100> : 0.1250+0.0000j => |01100|=0.1250 => Prob=0.0156
|01111> : -0.088+0.0884j => |01111|=0.1250 => Prob=0.0156
|01122> : 0.1250-0.0000j => |01122|=0.1250 => Prob=0.0156
|01133> : 0.0884-0.0884j => |01133|=0.1250 => Prob=0.0156
|02200> : -0.088+0.0884j => |02200|=0.1250 => Prob=0.0156
|02211> : 0.1250-0.0000j => |02211|=0.1250 => Prob=0.0156
|02222> : 0.0884-0.0884j => |02222|=0.1250 => Prob=0.0156
|02233> : 0.1250-0.0000j => |02233|=0.1250 => Prob=0.0156
|03300> : 0.1250-0.0000j => |03300|=0.1250 => Prob=0.0156
|03311> : 0.0884-0.0884j => |03311|=0.1250 => Prob=0.0156
|03322> : 0.1250-0.0000j => |03322|=0.1250 => Prob=0.0156
|03333> : -0.088+0.0884j => |03333|=0.1250 => Prob=0.0156
|10303> : 0.0000+0.1250j

## Test the partial trace on the subsystems to check the condition imposed for encryption operator

In [32]:
vect = vector_to_ket(qc.state)
rho = vect @ vect.conj().T #compute the density matrix of the state

qudit_index = 4 # Index of the qudit to keep - the sytem that we want to analyze
trace_out_indices = list(range(number_qudits)) # Indices of qudits to trace out
trace_out_indices.remove(qudit_index)

rho_S = partial_trace_qudits(rho, trace_out=trace_out_indices, num_total_qudits=number_qudits, dim_qudit=dimension)


In [33]:
print(f"Reduced density matrix on {qudit_index} subsystem:")
print_matrix(rho_S)

print("\n")

identity = (1/dimension)*gates.get_I_gate()
print(f"The resulting state is the maximally mixed state: {np.allclose(rho_S, identity)}")

Reduced density matrix on 4 subsystem:
0.25+0.0j 0.00+0.0j 0.00+0.0j 0.00+0.0j 
0.00+0.0j 0.25+0.0j 0.00+0.0j 0.00+0.0j 
0.00+0.0j 0.00+0.0j 0.25+0.0j 0.00+0.0j 
0.00+0.0j 0.00+0.0j 0.00+0.0j 0.25+0.0j 


The resulting state is the maximally mixed state: True
